# Data Quality Assessment

This notebook profiles and documents data quality for the Blade vs. Mallet IS477 course project.

It evaluates three datasets:

- **2025 PGA Tour SG: Putting**
- **2026 Masters Tournament leaderboard**
- **Integrated analysis dataset**

### Quality dimensions checked
- Completeness
- Validity
- Consistency
- Uniqueness
- Timeliness

### Output
- `logs/quality_report.txt`

In [1]:
import os
from datetime import datetime

import pandas as pd

In [2]:
# Folder paths
CLEAN_DIR = os.path.join(".", "data", "cleaned")
INT_DIR = os.path.join(".", "data", "integrated")
LOG_DIR = os.path.join(".", "logs")

os.makedirs(LOG_DIR, exist_ok=True)

# Store log lines so we can write a text report at the end
lines = []

In [3]:
def log(message=""):
    print(message)
    lines.append(str(message))


def section(title):
    log()
    log("=" * 60)
    log(f"  {title}")
    log("=" * 60)

In [4]:
def check_missing(df, name):
    log(f"\n[{name}] Completeness check:")
    total_rows = len(df)

    for col in df.columns:
        missing_count = df[col].isna().sum()
        missing_pct = 100 * missing_count / total_rows
        flag = " ⚠" if missing_count > 0 else ""
        log(f"  {col:<35} {missing_count:>3} missing  ({missing_pct:.1f}%){flag}")


def check_duplicates(df, key_cols, name):
    log(f"\n[{name}] Uniqueness check on {key_cols}:")
    duplicates = df.duplicated(subset=key_cols, keep=False)
    log(f"  Duplicate rows: {duplicates.sum()}")

    if duplicates.sum() > 0:
        log(df[duplicates][key_cols].to_string())


def check_vocab(df, col, valid_values, name):
    log(f"\n[{name}] Validity check – column '{col}':")
    counts = df[col].value_counts(dropna=False)
    log(counts.to_string())

    invalid_rows = df[~df[col].isin(valid_values)]
    status = " ⚠" if len(invalid_rows) > 0 else " ✓"
    log(f"  Invalid values: {len(invalid_rows)}{status}")

## Report header
This records the basic metadata for the quality report.

In [5]:
log("DATA QUALITY REPORT")
log("Project: Blade vs. Mallet – PGA Tour Putting Performance")
log(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')} local time")
log("Author: Jack Forman | IS477 SP26")

DATA QUALITY REPORT
Project: Blade vs. Mallet – PGA Tour Putting Performance
Generated: 2026-05-05 11:20 local time
Author: Jack Forman | IS477 SP26


## Dataset 1 — 2025 PGA Tour SG: Putting

In [6]:
section("DATASET 1 – 2025 PGA Tour SG:Putting")

pga = pd.read_csv(os.path.join(CLEAN_DIR, "pga_sgputt_2025_clean.csv"))

log(f"\nShape: {pga.shape[0]} rows × {pga.shape[1]} columns")
log(f"Season: {pga['season'].unique()}")
log("Source: PGA Tour official stats / Eden Steak analysis (edensteak.com)")
log("Access date: April 2026")
log("License: PGA Tour stats – publicly available; educational use")


  DATASET 1 – 2025 PGA Tour SG:Putting

Shape: 40 rows × 11 columns
Season: [2025]
Source: PGA Tour official stats / Eden Steak analysis (edensteak.com)
Access date: April 2026
License: PGA Tour stats – publicly available; educational use


In [7]:
check_missing(pga, "PGA 2025")
check_duplicates(pga, ["player_name", "season"], "PGA 2025")
check_vocab(pga, "putter_type", {"blade", "mallet", "unknown"}, "PGA 2025")


[PGA 2025] Completeness check:
  rank                                  0 missing  (0.0%)
  player_name                           0 missing  (0.0%)
  sg_putting_avg                        0 missing  (0.0%)
  putter_type                           0 missing  (0.0%)
  putter_brand                          0 missing  (0.0%)
  putter_model                          0 missing  (0.0%)
  putter_release_year                   0 missing  (0.0%)
  group                                 0 missing  (0.0%)
  season                                0 missing  (0.0%)
  source                                0 missing  (0.0%)
  putter_era                            0 missing  (0.0%)

[PGA 2025] Uniqueness check on ['player_name', 'season']:
  Duplicate rows: 0

[PGA 2025] Validity check – column 'putter_type':
putter_type
mallet    28
blade     12
  Invalid values: 0 ✓


In [8]:
log("\n[PGA 2025] Value range check – sg_putting_avg:")
log(f"  Min:  {pga['sg_putting_avg'].min():.3f}")
log(f"  Max:  {pga['sg_putting_avg'].max():.3f}")
log(f"  Mean: {pga['sg_putting_avg'].mean():.3f}")
log(f"  Std:  {pga['sg_putting_avg'].std():.3f}")

if pga["sg_putting_avg"].between(-1.5, 1.5).all():
    log("  Expected range for season averages: [-1.5, +1.5]  ✓")
else:
    log("  ⚠ Values outside expected range")

log("\n[PGA 2025] Coverage note:")
log("  Dataset covers only top 20 and bottom 20 players by SG:Putting.")
log("  The 120 middle-ranked players are not included in this dataset.")
log("  This is a known limitation: analysis reflects extremes, not the full distribution.")
log("  Ben Griffin (rank 19) switched from blade to mallet mid-season –")
log("  he is classified as blade per his dominant-season designation but")
log("  this introduces measurement uncertainty for ~1-3 players who switched.")


[PGA 2025] Value range check – sg_putting_avg:
  Min:  -0.970
  Max:  0.983
  Mean: 0.010
  Std:  0.607
  Expected range for season averages: [-1.5, +1.5]  ✓

[PGA 2025] Coverage note:
  Dataset covers only top 20 and bottom 20 players by SG:Putting.
  The 120 middle-ranked players are not included in this dataset.
  This is a known limitation: analysis reflects extremes, not the full distribution.
  Ben Griffin (rank 19) switched from blade to mallet mid-season –
  he is classified as blade per his dominant-season designation but
  this introduces measurement uncertainty for ~1-3 players who switched.


## Dataset 2 — 2026 Masters Tournament leaderboard

In [9]:
section("DATASET 2 – 2026 Masters Tournament Leaderboard")

masters = pd.read_csv(os.path.join(CLEAN_DIR, "masters_2026_clean.csv"))

log(f"\nShape: {masters.shape[0]} rows × {masters.shape[1]} columns")
log("Tournament: 2026 Masters, Augusta National, April 9-12 2026")
log("Source: Golf Monthly WITB articles, EssentiallySports, Sky Sports, Wikipedia")
log("Access date: April 13-14, 2026")
log("License: Publicly available equipment/results journalism; educational use")


  DATASET 2 – 2026 Masters Tournament Leaderboard

Shape: 44 rows × 13 columns
Tournament: 2026 Masters, Augusta National, April 9-12 2026
Source: Golf Monthly WITB articles, EssentiallySports, Sky Sports, Wikipedia
Access date: April 13-14, 2026
License: Publicly available equipment/results journalism; educational use


In [10]:
check_missing(masters, "Masters 2026")
check_duplicates(masters, ["player_name"], "Masters 2026")
check_vocab(masters, "putter_type", {"blade", "mallet", "unknown"}, "Masters 2026")


[Masters 2026] Completeness check:
  finish                                0 missing  (0.0%)
  player_name                           0 missing  (0.0%)
  score_to_par                          0 missing  (0.0%)
  r1                                    0 missing  (0.0%)
  r2                                    0 missing  (0.0%)
  r3                                    4 missing  (9.1%) ⚠
  r4                                    4 missing  (9.1%) ⚠
  putter_type                           0 missing  (0.0%)
  putter_brand                          0 missing  (0.0%)
  putter_model                          0 missing  (0.0%)
  putter_notes                          0 missing  (0.0%)
  source                                0 missing  (0.0%)
  finish_num                            0 missing  (0.0%)

[Masters 2026] Uniqueness check on ['player_name']:
  Duplicate rows: 0

[Masters 2026] Validity check – column 'putter_type':
putter_type
mallet    32
blade     12
  Invalid values: 0 ✓


## Integrated dataset
This section checks the merged dataset and compares overlap between the PGA and Masters data.

In [11]:
section("INTEGRATED DATASET")

integrated = pd.read_csv(os.path.join(INT_DIR, "integrated_putter_analysis.csv"))

log(f"\nShape: {integrated.shape[0]} rows × {integrated.shape[1]} columns")


  INTEGRATED DATASET

Shape: 75 rows × 16 columns


In [12]:
check_missing(integrated, "Integrated")

log("\n[Integrated] Player overlap between datasets:")
both = integrated["putter_type_2025"].notna() & integrated["putter_type_masters"].notna()
pga_only = integrated["putter_type_2025"].notna() & integrated["putter_type_masters"].isna()
masters_only = integrated["putter_type_2025"].isna() & integrated["putter_type_masters"].notna()

log(f"  In both datasets: {both.sum()} players")
log(f"  PGA 2025 only:    {pga_only.sum()} players")
log(f"  Masters only:     {masters_only.sum()} players")


[Integrated] Completeness check:
  player_name                           0 missing  (0.0%)
  pga_2025_rank                        35 missing  (46.7%) ⚠
  pga_2025_sg_putt                     35 missing  (46.7%) ⚠
  putter_type_2025                     35 missing  (46.7%) ⚠
  putter_brand_2025                    35 missing  (46.7%) ⚠
  putter_model_2025                    35 missing  (46.7%) ⚠
  pga_2025_group                       35 missing  (46.7%) ⚠
  season                               35 missing  (46.7%) ⚠
  masters_2026_finish                  35 missing  (46.7%) ⚠
  masters_2026_finish_num              35 missing  (46.7%) ⚠
  masters_2026_score                   35 missing  (46.7%) ⚠
  putter_type_masters                  35 missing  (46.7%) ⚠
  putter_brand_masters                 35 missing  (46.7%) ⚠
  putter_model_masters                 35 missing  (46.7%) ⚠
  putter_consistency                    0 missing  (0.0%)
  putter_type_unified                   0 missing  (0.0%)

In [13]:
players_in_both = integrated[both]["player_name"].tolist()

log(f"\n  Players appearing in both datasets: {players_in_both}")
log("  (These players allow direct cross-dataset comparison)")

log("\n[Integrated] Putter type consistency for shared players:")
shared = integrated[both][
    ["player_name", "putter_type_2025", "putter_type_masters", "putter_consistency"]
]

log(shared.to_string(index=False))


  Players appearing in both datasets: ['Cameron Young', 'Jake Knapp', 'Rory Mcilroy', 'Sam Burns', 'Tommy Fleetwood']
  (These players allow direct cross-dataset comparison)

[Integrated] Putter type consistency for shared players:
    player_name putter_type_2025 putter_type_masters putter_consistency
  Cameron Young           mallet              mallet         consistent
     Jake Knapp           mallet              mallet         consistent
   Rory Mcilroy           mallet              mallet         consistent
      Sam Burns           mallet              mallet         consistent
Tommy Fleetwood           mallet              mallet         consistent


## Overall quality summary
This is the plain-language wrap-up of the project’s main data quality findings.

In [14]:
section("OVERALL QUALITY SUMMARY")

log("""
DIMENSION        ASSESSMENT
─────────────────────────────────────────────────────────
Completeness     PGA 2025: 100% complete (no missing values)
                 Masters 2026: r3/r4 null for 4 MC players (expected)
                 Integrated: partial overlap by design (different populations)

Validity         All putter_type values are valid controlled vocabulary
                 (blade / mallet / unknown). SG:Putting values within
                 expected real-world range for season averages.

Consistency      5 players appear in both datasets; all 5 show consistent
                 putter type classifications across sources. No contradictions
                 detected. One known mid-season switcher (Ben Griffin)
                 is documented with a note.

Uniqueness       No duplicate player records in any dataset.

Timeliness       PGA 2025 data reflects completed 2025 FedExCup season.
                 Masters 2026 data collected April 13-14, 2026,
                 immediately after tournament conclusion.

Coverage         Intentional limitation: PGA 2025 covers only top/bottom 20
                 players (extremes). Masters 2026 covers 44 players
                 (full field cut + 4 MC players). Future work should
                 expand PGA dataset to cover all ~180 ranked players.

Provenance       All sources documented per-row in source column.
                 Multiple corroborating sources used for top finishers.
                 Lower-confidence records flagged in notes column.
""")


  OVERALL QUALITY SUMMARY

DIMENSION        ASSESSMENT
─────────────────────────────────────────────────────────
Completeness     PGA 2025: 100% complete (no missing values)
                 Masters 2026: r3/r4 null for 4 MC players (expected)
                 Integrated: partial overlap by design (different populations)

Validity         All putter_type values are valid controlled vocabulary
                 (blade / mallet / unknown). SG:Putting values within
                 expected real-world range for season averages.

Consistency      5 players appear in both datasets; all 5 show consistent
                 putter type classifications across sources. No contradictions
                 detected. One known mid-season switcher (Ben Griffin)
                 is documented with a note.

Uniqueness       No duplicate player records in any dataset.

Timeliness       PGA 2025 data reflects completed 2025 FedExCup season.
                 Masters 2026 data collected April 13-14, 2026,
 

In [15]:
log_path = os.path.join(LOG_DIR, "quality_report.txt")

with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"\nQuality report written → {log_path}")


Quality report written → .\logs\quality_report.txt
